# Lab 9: Transfer Learning with BERT

In this lab we are going to be looking at fine-tuning a BERT model to carry out a sequence classification task.

We are going to load in some text from a small number of books in the Gutenberg corpus and see if we can train a classifier to classify which book a piece of text is from 

First lets pull up all of the filenames available.

In [3]:
import os,random,math
TRAINING_DIR="../lab_2_ngrams/sentence-completion/Holmes_Training_Data" #this needs to be the parent directory for the training corpus

filenames=os.listdir(TRAINING_DIR)
n=len(filenames)
print("There are {} files in the training directory: {}".format(n,TRAINING_DIR))

#print(filenames)

There are 523 files in the training directory: ../lab_2_ngrams/sentence-completion/Holmes_Training_Data


We are going to create a Book class to store the text from a class and do some very basic pre-processing.  We need to 
* load in the text line-by-line
* get rid of header lines (the stuff in the file before the line which starts \*END\*THE SMALL PRINT!)
* make chunks of text which are longer than 1 line.  These should be easier to classify.  We will try to get sentences - but some chunks may contain multiple sentences.  We are not going to worry about this here.
* return some labelled data split randomly between training and testing

In [6]:
class Book():
    TRAINING_DIR="../lab_2_ngrams/sentence-completion/Holmes_Training_Data"
    header_end="*END*THE SMALL PRINT!"
    seed=53
   
    def __init__(self,filename,label=""):
        self.filename=filename
        self.loadfile()
        self.make_chunks()
        if label=="":
            self.label=self.filename
        else:
            self.label=label
        
    def loadfile(self):
        filepath=os.path.join(Book.TRAINING_DIR,self.filename)
        self.lines=[]
        beyond_header=False
        try:
            with open(filepath) as instream:
                for line in instream:
                    line=line.rstrip()
                    
                    if len(line)>0 and beyond_header:
                        self.lines.append(line)
                    if line.startswith(Book.header_end):
                        beyond_header=True
        except UnicodeDecodeError:
            print(f"UnicodeDecodeError processing {filepath}")
    
    def length(self):
        return len(self.chunks)
    
    def head(self,n=10):
        return self.chunks[:n]
    
    def make_chunks(self):
        self.chunks=[]
        
        current=""
        for line in self.lines:
            current+=line+" "
            if line.endswith("."):
                self.chunks.append(current.rstrip())
                current=""
            
    def get_labelled_data(self,split=0.8):
        labelled_data=[(chunk,self.label) for chunk in self.chunks]
        random.seed(Book.seed)
        random.shuffle(labelled_data)
        index=int(self.length()*split)
        return (labelled_data[:index],labelled_data[index:])
                

### Exercise 1
- Create an instance of a Book() and store it in the variable `emma`.  The filename is `EMMA10.TXT` and the label should be `Emma`
- Check the number of sentences = 2028 and have a look at the first 10 sentence
- Repeat to create `ivanhoe` to store the text from `IVNHO12.TXT` with the label `Ivanhoe`.  
- The number of sentences in `ivanhoe` should be 1743

In [7]:
emma=Book("EMMA10.TXT","Emma")

In [8]:
emma.length()

2028

In [9]:
emma.head()

['VOLUME I CHAPTER I Emma Woodhouse, handsome, clever, and rich, with a comfortable home and happy disposition, seemed to unite some of the best blessings of existence; and had lived nearly twenty-one years in the world with very little to distress or vex her.',
 "She was the youngest of the two daughters of a most affectionate, indulgent father; and had, in consequence of her sister's marriage, been mistress of his house from a very early period.  Her mother had died too long ago for her to have more than an indistinct remembrance of her caresses; and her place had been supplied by an excellent woman as governess, who had fallen little short of a mother in affection.",
 "Sixteen years had Miss Taylor been in Mr. Woodhouse's family, less as a governess than a friend, very fond of both daughters, but particularly of Emma.  Between them it was more the intimacy of sisters.  Even before Miss Taylor had ceased to hold the nominal office of governess, the mildness of her temper had hardly a

In [10]:
ivanhoe=Book("IVNHO12.TXT","Ivanhoe")

In [11]:
ivanhoe.length()

1743

In [12]:
ivanhoe.head(10)

['IVANHOE.',
 "CHAPTER I Thus communed these; while to their lowly dome, The full-fed swine return'd with evening home; Compell'd, reluctant, to the several sties, With din obstreperous, and ungrateful cries.",
 "               Pope's _Odyssey_.",
 'In that pleasant district of merry England which is watered by the river Don, there extended in ancient times a large forest, covering the greater part of the beautiful hills and valleys which lie between Sheffield and the pleasant town of Doncaster.  The remains of this extensive wood are still to be seen at the noble seats of Wentworth, of Warncliffe Park, and around Rotherham.  Here haunted of yore the fabulous Dragon of Wantley; here were fought many of the most desperate battles during the Civil Wars of the Roses; and here also flourished in ancient times those bands of gallant outlaws, whose deeds have been rendered so popular in English song.',
 'Such being our chief scene, the date of our story refers to a period towards the end of 

### Exercise 2
- Use the `get_labelled_data()` method to get a training and testing portion from each book (split = 80%).  
- Create 2 pandas dataframes
    - 1 dataframe called `training_df` with all of the training data (from both books)
    - 1 dataframe called  `testing_df`  with all of the test data (from both books).  
    - The columns of both dataframes should be `text` and `label`

In [13]:
(training1,testing1)=emma.get_labelled_data(0.8)

In [14]:
training1[:5]

[('I made the match, you know, four years ago; and to have it take place, and be proved in the right, when so many people said Mr. Weston would never marry again, may comfort me for any thing." Mr. Knightley shook his head at her.  Her father fondly replied, "Ah! my dear, I wish you would not make matches and foretell things, for whatever you say always comes to pass.  Pray do not make any more matches." "I promise you to make none for myself, papa; but I must, indeed, for other people.  It is the greatest amusement in the world! And after such success, you know!--Every body said that Mr. Weston would never marry again.  Oh dear, no! Mr. Weston, who had been a widower so long, and who seemed so perfectly comfortable without a wife, so constantly occupied either in his business in town or among his friends here, always acceptable wherever he went, always cheerful-- Mr. Weston need not spend a single evening in the year alone if he did not like it.  Oh no! Mr. Weston certainly would neve

In [15]:
(training2,testing2)=ivanhoe.get_labelled_data(0.8)

In [16]:
training=training1+training2
testing=testing1+testing2
random.shuffle(training)
random.shuffle(testing)

In [17]:
training[:5]

[('I am only afraid of your sitting up for me.  I am not afraid of your not being exceedingly comfortable with Mrs. Goddard.',
  'Emma'),
 ("CHAPTER XXXIII ---Flower of warriors, How is't with Titus Lartius? _Marcius_. As with a man busied about decrees, Condemning some to death and some to exile, Ransoming him or pitying, threatening the other.",
  'Ivanhoe'),
 ('Much, much beyond impropriety!--It has sunk him, I cannot say how it has sunk him in my opinion.  So unlike what a man should be!-- None of that upright integrity, that strict adherence to truth and principle, that disdain of trick and littleness, which a man should display in every transaction of his life." "Nay, dear Emma, now I must take his part; for though he has been wrong in this instance, I have known him long enough to answer for his having many, very many, good qualities; and--" "Good God!" cried Emma, not attending to her.--"Mrs. Smallridge, too! Jane actually on the point of going as governess!  What could he mean

In [18]:
import pandas as pd

training_df=pd.DataFrame(training,columns=["text","label"])
testing_df=pd.DataFrame(testing,columns=["text","label"])

training_df.head()

,text,label
0,I am only afraid of your sitting up for me. I...,Emma
1,"CHAPTER XXXIII ---Flower of warriors, How is't...",Ivanhoe
2,"Much, much beyond impropriety!--It has sunk hi...",Emma
3,I know what you mean--but Emma's hand is the s...,Emma
4,"---Come on you, too, my masters, tarry not to ...",Ivanhoe


## Finetuning BERT to tell the difference between sentences from each book

Now we are going to look at building a classifier on top of BERT.  The first thing we need to do is map the informative label names we have (`Emma` and `Ivanhoe`) to integers which will be used by BERT.  In this simple case, we could just create a dictionary manually.  However, the code below will make a sorted list (without duplicates) of all of the labelnames in the two dataframes and then generate a dictionary which maps each label name to an integer.


In [19]:
#first we need a map for the labels

#make a list of all of the unique labels in the training and testing dataframes
#sorting it means that it will also be in the same order (alphabetical) rather than depending on order in the training / testing data
labellist=sorted(list(set(training_df['label'].unique()).union(set(testing_df['label'].unique())))) 

labels={label:i for i,label in enumerate(labellist)}
labels

{'Emma': 0, 'Ivanhoe': 1}

### Exercise 3
Write some code to create a reverse index for the labels which maps the numbers back to the more informative strings

This should result in something which looks as follows:

```
reverse_index={0:'Emma',1:'Ivanhoe'}
```

But obviously, you should create it automatically from the labels dictionary rather than typing it in!

In [20]:
reverse_index={value:key for (key,value)in labels.items()}

In [21]:
reverse_index

{0: 'Emma', 1: 'Ivanhoe'}

Now we need to store the data in a Dataset class.  This inherits properties from torch's Dataset class and is what is expected.  It is initialised by giving it dataframe from which it can extract a list of labels and a list of texts.   It handles preprocessing including adding CLS and SEP tokens at the beginning and end, tokenization, lower-casing truncation and padding.  It also provides a `\_\_getitem\_\_()` method which allows the Dataset to be indexed into like a list i.e., `myDataset[3]` will return a pair which is the label and text with index 3.

In [22]:
import torch
import numpy as np
from transformers import BertTokenizer
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')

class Dataset(torch.utils.data.Dataset):
    
    def __init__(self,df,column='text'):
        self.labels=[labels[label] for label in df['label']]
        self.texts=[tokenizer(text.lower(),padding='max_length',max_length=512,truncation=True,return_tensors="pt") for text in df[column]]
        
    def classes(self):
        return self.labels
    
    def __len__(self):
        return len(self.labels)
    
    def get_batch_labels(self,idx):
        return np.array(self.labels[idx])
    
    def get_batch_texts(self,idx):
        return self.texts[idx]
    
    def __getitem__(self,idx):
        batch_texts=self.get_batch_texts(idx)
        batch_y=self.get_batch_labels(idx)
        
        return batch_texts,batch_y
    

train_data=Dataset(training_df)
test_data=Dataset(testing_df)

/Users/lukebirkett/Repos/study-planner/968G5_Advanced_Natural_Language_Processing/adv_nlp/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Lets have a look at one of the dataset items.

The \_\_getitem\_\_ method allows us to index into the dataset and get a particular item

In [27]:
train_data[0]

<method-wrapper '__getitem__' of tuple object at 0x1251dc580>

We can also just look at the text (or label) for an item as follows.  Note that the tokens have been replaced by their indices in the BERT wordpiece vocabulary.

In [28]:
train_data.texts[0]

{'input_ids': tensor([[  101,  1045,  2572,  2069,  4452,  1997,  2115,  3564,  2039,  2005,
          2033,  1012,  1045,  2572,  2025,  4452,  1997,  2115,  2025,  2108,
         17003,  2135,  6625,  2007,  3680,  1012, 22547,  1012,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,  

### Exercise 4
Can you turn the token ids back into subwords for one of the inputs?

In [29]:
my_input = train_data.texts[0].input_ids
my_input

tensor([[  101,  1045,  2572,  2069,  4452,  1997,  2115,  3564,  2039,  2005,
          2033,  1012,  1045,  2572,  2025,  4452,  1997,  2115,  2025,  2108,
         17003,  2135,  6625,  2007,  3680,  1012, 22547,  1012,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,  

In [30]:
my_tokens=tokenizer.convert_ids_to_tokens(my_input[0])
my_tokens

['[CLS]',
 'i',
 'am',
 'only',
 'afraid',
 'of',
 'your',
 'sitting',
 'up',
 'for',
 'me',
 '.',
 'i',
 'am',
 'not',
 'afraid',
 'of',
 'your',
 'not',
 'being',
 'exceeding',
 '##ly',
 'comfortable',
 'with',
 'mrs',
 '.',
 'goddard',
 '.',
 '[SEP]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PAD]',
 '[PA

Now we need to prepare the inputs for the particular device (GPU or CPU) that the model is going to be run on.  Let's just check first whether GPU / CUDA has been enabled.

In [31]:
use_cuda=torch.cuda.is_available()
if use_cuda:
  print("GPU acceleration enabled")
else:
  print("GPU acceleration NOT enabled.  If using Colab, have you changed the runtype type and selected GPU as the hardware accelerator?")
device=torch.device("cuda" if use_cuda else "cpu")

GPU acceleration NOT enabled.  If using Colab, have you changed the runtype type and selected GPU as the hardware accelerator?


In [33]:
def prepare_inputs(input1,label,device):
  label=label.to(device)
  mask=input1['attention_mask'].to(device)
  input_id=input1['input_ids'].squeeze(1).to(device)
  return (input_id,mask,label)

Lets try preparing some inputs and running them through BERT.  We will use the torch DataLoader to manage iterating over the datasets during training and testing.  Here we will just process the first item produced by the DataLoader to see the output from the pre-trained BERT model.

In [34]:
from transformers import BertModel

train_dataloader=torch.utils.data.DataLoader(train_data,batch_size=2,shuffle=True)
bert=BertModel.from_pretrained('bert-base-uncased')
for train_input,train_label in train_dataloader:
    input_id,mask,label=prepare_inputs(train_input,train_label,device)
    output=bert(input_ids=input_id,attention_mask=mask,return_dict=False)
    break

print(input_id,mask,label)
print(len(output))
output[1]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14236.90it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


IndexError: index 1 is out of bounds for dimension 1 with size 1

Now, we need to construct our classification network.  Look at the code below and then answer Exercise 5

In [35]:
#now we need to put a simple classification layer on top of BERT

from torch import nn
from transformers import BertModel

class BertClassifier(nn.Module):
    
    def __init__(self,dropout=0.5,num_classes=2):
        
        super(BertClassifier,self).__init__()
        
        self.bert=BertModel.from_pretrained('bert-base-uncased')
        self.dropout=nn.Dropout(dropout)
        self.linear=nn.Linear(768,num_classes)
        self.relu=nn.ReLU()
        
    def forward(self,input_id,mask):
        
        last_hidden_layer,pooled_output = self.bert(input_ids=input_id,attention_mask=mask,return_dict=False)
        dropout_output=self.dropout(pooled_output)
        linear_output=self.linear(dropout_output)
        final_layer=self.relu(linear_output)
        
        return final_layer

### Exercise 5

Use the definitions of the initialisation method and the forward method of the BertClassifier class to sketch out what the neural network architecture looks like.

What do you understand by the terms:

- pooled output

- dropout layer

- linear layer

- relu layer


Now we will define a function which will carry out the training of the network.  It will handle
- setting up the DataLoaders
- preparing the inputs for CPU / GPU
- carrying out training and validation for a number of epochs.  In each epoch:
    - iterate over the training data in batches
    - get the output for each input and compute the batch loss (Cross Entropy)
    - use the batch loss to carry out optimisation
    - compute the accuracy and total loss for the training data
    - iterate over the testing / validation data in batches
    - get the output for each input and compute the batch loss
    - compute the accuracy and total loss for the validation data
    - output stats

In [30]:
#we now need a training loop

from torch.optim import Adam
from tqdm import tqdm



def train(model, train_data,val_data,learning_rate,epochs):
    
    train_dataloader=torch.utils.data.DataLoader(train_data,batch_size=2,shuffle=True)
    val_dataloader=torch.utils.data.DataLoader(test_data,batch_size=2)
    
    use_cuda=torch.cuda.is_available()
    device=torch.device("cuda" if use_cuda else "cpu")
    
    criterion=nn.CrossEntropyLoss()
    optimizer=Adam(model.parameters(),lr=learning_rate)
    
    if use_cuda:
        model=model.cuda()
        criterion=criterion.cuda()
        
    for epoch_num in range(epochs):
        total_acc_train=0
        total_loss_train=0
        model.train()
        for train_input,train_label in tqdm(train_dataloader):
            
            input_id,mask, train_label=prepare_inputs(train_input,train_label,device)
            
            output=model(input_id,mask)
            
            batch_loss=criterion(output,train_label.long())
            total_loss_train +=batch_loss.item()
            
            acc=(output.argmax(dim=1)==train_label).sum().item()
            total_acc_train+=acc
            
            model.zero_grad()
            batch_loss.backward()
            optimizer.step()
            
        total_acc_val=0
        total_loss_val=0
        model.eval()
        with torch.no_grad():
            for val_input,val_label in val_dataloader:
                
                input_id,mask, val_label=prepare_inputs(val_input,val_label,device)
                
                output=model(input_id,mask)
                
                batch_loss=criterion(output,val_label.long())
                
                total_loss_val+=batch_loss.item()
                
                acc=(output.argmax(dim=1)==val_label).sum().item()
                total_acc_val+=acc
                
        print(f'Epochs: {epoch_num+1} | Train Loss: {total_loss_train / len(train_data):.3f} | Train Accuracy: {total_acc_train/len(train_data):.3f}')
        print(f'Val loss: {total_loss_val/len(val_data):.3f} | Val Accuracy: {total_acc_val / len(val_data):.3f}')
        

Here, we define the number of epochs we are going to train for, the learning rate and an instance of our BertClassifier network.

In [31]:
EPOCHS=1  
model=BertClassifier(num_classes=len(labels.keys()))
LR=1e-6


In [32]:
model=BertClassifier(num_classes=len(labels.keys()))

#this will freeze the pre-trained BERT model and just make the classification head trainable
#can speed things up and avoid "catastrophic forgetting" / overfitting on task-specific data
model.bert.requires_grad_(False) 

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0): BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          

Now we are actually going to train the model.  This might take some time - particularly if you are running on a CPU (1.5 hrs per epoch on my laptop!).  I recommend running on colab with GPU enabled (or using the lab machines with GPU).

In [33]:
train(model,train_data,test_data,LR,EPOCHS)

  1%|▌                                        | 20/1508 [00:22<28:27,  1.15s/it]


KeyboardInterrupt: 

We need to be able to save the model to be able to use it elsewhere (without training again!)

In [34]:
#I am not going to save my model as I didn't wait for it to finish training!

#output_dir="bert-base-uncased-bookclassifier"
#torch.save(model,output_dir)

We can load it up like this.  This could be in another notebook.  If loading in another notebook, you should make sure the BertClassifier class is also defined in that notebook (along with other necessary imports).

In [35]:
!pwd

/Users/juliewe/Dropbox/teaching/AdvancedNLP/2026/2026_content/week9/lab9


In [36]:
#I am going to load a model I trained on colab!

input_dir="Lab9resources/julie-bert-base-uncased-bookclassifier"
complete_model=torch.load(input_dir)

Here's an evaluation loop we can use to evaluate on some test data.  We also return the predictions which can than be added to the dataframe

In [37]:
batchsize=2
def evaluate(model,test_dataset):
    model.eval()
    test_dataloader=torch.utils.data.DataLoader(test_dataset,batch_size=batchsize)
    
    use_cuda=torch.cuda.is_available()
    device=torch.device("cuda" if use_cuda else "cpu")
    
    if use_cuda:
        model=model.cuda()
        
    total_acc_test=0
    with torch.no_grad():
        count=0
        predictions=[]
        for test_input,test_label in tqdm(test_dataloader):
            count+=batchsize
            test_label=test_label.to(device)
            mask=test_input['attention_mask'].to(device)
            input_id=test_input['input_ids'].squeeze(1).to(device)
            output=model(input_id,mask)
            #print(output.argmax(dim=1),test_label)
            predictions.append(output.argmax(dim=1))  #save the prediction for further analysis
            acc=(output.argmax(dim=1)==test_label).sum().item()
            
            total_acc_test+=acc
            if count%100==0:
                print(f'Accuracy so far = {total_acc_test/count: .3f}')
            
    print(f'Test accuracy: {total_acc_test/len(test_dataset): .3f}')
    return predictions

This takes around 6 minutes to run on my laptop.  I have the evaluation loop printing out "Accuracy so far" as I get bored waiting to the end to see the results - it also gives a very rough indication of how stable the method is

In [38]:
predictions=evaluate(model, test_data)

 13%|█████▌                                    | 50/378 [00:54<05:56,  1.09s/it]

Accuracy so far =  0.620


 26%|██████████▊                              | 100/378 [01:49<05:06,  1.10s/it]

Accuracy so far =  0.650


 40%|████████████████▎                        | 150/378 [02:44<04:08,  1.09s/it]

...

Accuracy so far =  0.645


 93%|█████████████████████████████████████▉   | 350/378 [06:18<00:28,  1.02s/it]

Accuracy so far =  0.650


100%|█████████████████████████████████████████| 378/378 [06:46<00:00,  1.08s/it]

Test accuracy:  0.649


### Exercise 6
Add the predicted label for each test item to the dataframe with the test data.

In [39]:
flattened=[]
for batch in predictions:
    for pred in batch:
        flattened.append(reverse_index[pred.item()])
testing_df['prediction']=flattened
testing_df.head(50)

,text,label,prediction
0,"About the same time arrived Cedric the Saxon, ...",Ivanhoe,Emma
1,NOTE TO CHAPTER XXI.,Ivanhoe,Emma
2,But the moment had now arrived when earth and ...,Ivanhoe,Ivanhoe
3,"But the expression is hardly admissible, Mrs. ...",Emma,Emma
4,"Seest thou her locks, whose sunny glow Half sh...",Ivanhoe,Emma
5,---But thy words have awakened a new soul with...,Ivanhoe,Emma
6,"Colonel Campbell is a very agreeable man, and ...",Emma,Emma
7,This was no longer matter of so much difficult...,Ivanhoe,Emma
8,His own education had taught him no skill in t...,Ivanhoe,Emma
9,"To look at her, nobody would think how delight...",Emma,Ivanhoe


### Exercise 7
Compute the confusion matrix / precision and recall scores for the different classes.  What does this analysis tell you about the errors?

In [40]:
all_labels=testing_df['label']
all_predictions=testing_df['prediction']

In [41]:
tp={}
fp={}
fn={}
tn={}

for label1,pred1 in zip(all_labels,all_predictions):
    for label in labels.keys():
        if label1==label:
            if pred1==label:
                tp[label]=tp.get(label,0)+1
            else:
                fn[label]=fn.get(label,0)+1
            
        else:
            if pred1==label:
                fp[label]=fp.get(label,0)+1
            else:
                tn[label]=tn.get(label,0)+1
                


precision={label:value/(value+fp.get(label,0)) for label,value in tp.items()}
recall={label:value/(value+fn.get(label,0)) for label,value in tp.items()}
f1={label:(2*value*recall.get(label,0))/(value+recall.get(label,0)) for label,value in precision.items()}
                

In [42]:
precision

{'Ivanhoe': 0.8387096774193549, 'Emma': 0.6117274167987322}

In [43]:
recall

{'Ivanhoe': 0.2979942693409742, 'Emma': 0.9507389162561576}

In [44]:
f1

{'Ivanhoe': 0.43974630021141653, 'Emma': 0.7444551591128254}

Precision is better for Ivanhoe whereas recall is better for Emma.  This suggests that the classifier is over-predicting Emma - this is consistent with there being more training data for Emma.

## Extension
Now carry out Lab 9b on sequence labelling with BERT.
